> **Research record.** Cleaned copy of `900-features-stat-areas/110-create-dataset-quarterly.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Statistical area x quarter dataset (for each area the best-covered month of the quarter), plus home-to-work commuting distance.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

In [ ]:
import pandas as pd
from geofunctions import utils
import geopandas as gpd

In [ ]:
MONTHLY_DATASET_TABLE = "phd_datasets.sa_dataset_monthly_20250523"

In [ ]:
QUARTERLY_DATASET_TABLE= "phd_datasets.sa_dataset_quarterly"

### New version where we choose best month in the quarter individually for each stat_area and we keep full grid

In [ ]:
DATE='20250609'

In [ ]:
MONTHLY_DATASET_TABLE = "phd_datasets.sa_dataset_monthly_20250523"

In [ ]:
utils.read_query_to_dataframe(f"""CREATE OR REPLACE TABLE {QUARTERLY_DATASET_TABLE}_{DATE} AS
WITH all_stat_areas AS (
  SELECT DISTINCT CAST(yishuv_stat_2022 AS STRING) AS yishuv_stat_2022
  FROM `phd-habidatum.phd_datasets.sa_neigborhoods_types`
),

all_quarters AS (
  SELECT DISTINCT
    CONCAT(CAST(EXTRACT(YEAR FROM month) AS STRING), 'Q', CAST(EXTRACT(QUARTER FROM month) AS STRING)) AS quarter
  FROM `phd-habidatum.phd_datasets.remote_work_agg_sa_20250321`
  
),

full_grid AS (
  SELECT *
  FROM all_stat_areas
  CROSS JOIN all_quarters
),

t AS (
  SELECT 
    CAST(b.yishuv_stat_2022 AS STRING) AS yishuv_stat_2022,
    month,
    CONCAT(CAST(EXTRACT(YEAR FROM month) AS STRING), 'Q', CAST(EXTRACT(QUARTER FROM month) AS STRING)) AS quarter,
    COUNTIF(num_homes >= 3) AS count_min3
  FROM `phd-habidatum.phd_datasets.remote_work_agg_sa_20250321` b
  -- where DATE(month)< DATE '2023-09-01'
  GROUP BY yishuv_stat_2022, month
),

ranked_months AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY quarter, yishuv_stat_2022 ORDER BY count_min3 DESC NULLS LAST) AS rn
  FROM t
),

best_months AS (
  SELECT quarter, yishuv_stat_2022, month AS selected_month
  FROM ranked_months
  WHERE rn = 1
),

final_selection AS (
  SELECT 
    g.quarter,
    g.yishuv_stat_2022,
    bm.selected_month
  FROM full_grid g
  LEFT JOIN best_months bm
    ON g.quarter = bm.quarter AND g.yishuv_stat_2022 = bm.yishuv_stat_2022
)

SELECT 
  fs.yishuv_stat_2022,
  fs.quarter,
  fs.selected_month,
  m.* EXCEPT(month,yishuv_stat_2022),
  
FROM final_selection fs
LEFT JOIN {MONTHLY_DATASET_TABLE} m
  ON fs.selected_month = m.month AND fs.yishuv_stat_2022 = cast(m.yishuv_stat_2022 as string)
"""
)

### Add commuting distance from 01/2020 to workplace

In [ ]:
WORK_DIST_TABLE = "phd-habidatum.phd_datasets.home_work_commuting_distance_sa"

In [ ]:
utils.read_query_to_dataframe(f"""
create or replace table {QUARTERLY_DATASET_TABLE}_{DATE} as
with r as (
SELECT yishuv_stat_2022_home yishuv_stat_2022 ,avg(distance_euclidean)as commuting_distance, sum(distance_euclidean*days)/sum(days) commuting_distance_wgt_days
FROM {WORK_DIST_TABLE}
where days>1  and month='202001'
group by yishuv_stat_2022_home
)
select qdf.*except(commuting_distance, commuting_distance_wgt_days), r.commuting_distance, r.commuting_distance_wgt_days
from {QUARTERLY_DATASET_TABLE}_{DATE} qdf
left join r
on qdf.yishuv_stat_2022=cast(r.yishuv_stat_2022 as string)
 """)

In [ ]:
df = utils.read_query_to_dataframe(f"""
SELECT
  s.yishuv_stat_2022, s.geometry, t.commuting_distance, t.commuting_distance_wgt_days,
  ST_DISTANCE(
    ST_CENTROID(s.geometry),
    ST_GEOGPOINT(34.79188915373317, 32.07472355010219)
  ) AS distance_to_azrieli,
from {QUARTERLY_DATASET_TABLE}_{DATE} t
join phd-habidatum.phd_adm.statistical_areas_gushdan s
using(yishuv_stat_2022) 
where quarter='2020Q1'""")